# Spark User Guide - Fabric Runtime 2.0

Fabric-native edition. The `spark` session and `SparkContext` are managed by Microsoft Fabric; do not create or stop them manually. Before running, select **Fabric Runtime 2.0** and attach a Lakehouse that you can write to. Any workspace and Lakehouse are supported; this checked-in notebook contains no tenant-specific IDs. The first code cell verifies Apache Spark 4.1 and prints the active Fabric context.


# Spark User Guide — comprehensive runnable PySpark workflows

This notebook implements the official **PySpark 4.1 User Guide** as a
production-oriented, executable course. It follows the official chapters:

1. [DataFrames](https://spark.apache.org/docs/latest/api/python/user_guide/dataframes.html)
2. [PySpark data types](https://spark.apache.org/docs/latest/api/python/user_guide/touroftypes.html)
3. [Data manipulation](https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html)
4. [Debugging PySpark](https://spark.apache.org/docs/latest/api/python/user_guide/bugbusting.html)
5. [UDFs and UDTFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html)
6. [SQL with PySpark](https://spark.apache.org/docs/latest/api/python/user_guide/sql.html)
7. [Data loading and storage](https://spark.apache.org/docs/latest/api/python/user_guide/loadandbehold.html)
8. [pandas API on Spark ANSI migration](https://spark.apache.org/docs/latest/api/python/user_guide/ansi_migration_guide.html)

These links are the current PySpark 4.1 source pages. The notebook adds
production contracts, assertions, and Microsoft Fabric Spark deployment diagnostics
while preserving their topic hierarchy.

This Fabric notebook runs through Microsoft Fabric Spark. Every code cell is bounded and
runnable in that mode. Driver-only, long-running, or optional facilities are
taught through capability checks and links to official documentation.


## 0. Start a remote session and understand the execution boundary

PySpark builds lazy logical plans in Python; the Spark server analyzes,
optimizes, and executes them. Microsoft Fabric Spark intentionally does not expose
`SparkContext` or direct RDD access. This separation makes DataFrame plans
portable across Python, SQL, Scala, R, TypeScript, Ruby, and Rust clients.


In [ ]:
# Microsoft Fabric injects the Spark session.
import importlib.util
import os
import sys

import pyspark
from notebookutils import runtime
from pyspark.sql import Row, Window, functions as F, types as T

spark = globals()["spark"]
fabric_context = runtime.context
actual_environment_id = fabric_context.get("environmentId")
assert spark.version.startswith("4.1"), (
    f"Fabric Runtime 2.0 requires Apache Spark 4.1; got {spark.version}"
)
print({
    "fabric_workspace_id": fabric_context.get("currentWorkspaceId"),
    "fabric_notebook": fabric_context.get("currentNotebookName"),
    "environment_id": actual_environment_id,
    "spark": spark.version,
    "application_id": spark.sparkContext.applicationId,
})

import os
import shutil
import uuid
from pathlib import Path

print({
    "client": pyspark.__version__,
    "server": spark.version,
    "fabric_runtime": "2.0",
})


# Chapter 1 — DataFrames: a view into structured data

A DataFrame is a distributed, schema-bearing logical relation. It resembles
a table but remains lazy until an action. The optimizer can reorder and
combine transformations because expressions are declarative.


## 1.1 Create a DataFrame from local records

Local Python values are useful for small dimensions and tests. In production,
avoid creating large DataFrames on the driver; read distributed storage.


In [ ]:
people = spark.createDataFrame([
    {"person_id": 1, "name": "Ada", "team": "platform"},
    {"person_id": 2, "name": "Grace", "team": "analytics"},
    {"person_id": 3, "name": "Linus", "team": "platform"},
])
assert people.count() == 3
assert set(people.columns) == {"person_id", "name", "team"}
people.orderBy("person_id").show()


## 1.2 Make schemas contractual

Explicit schemas control column order, widths, nullability, nested types,
and compatibility at system boundaries. Inferred schemas are convenient for
exploration but can drift when source values change.


In [ ]:
people_schema = T.StructType([
    T.StructField("person_id", T.LongType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("team", T.StringType(), True),
])
typed_people = spark.createDataFrame([
    (1, "Ada", "platform"),
    (2, "Grace", "analytics"),
    (3, "Linus", "platform"),
], people_schema)
assert typed_people.schema == people_schema
typed_people.printSchema()


## 1.3 Create deterministic typed fixtures with SQL

`VALUES` is concise for decimal, temporal, array, map, and null fixtures.
Temporary views let SQL-created data flow into DataFrame APIs.


In [ ]:
spark.sql("""CREATE OR REPLACE TEMP VIEW guide_orders AS
SELECT *
  FROM VALUES
  (1001, 1, TIMESTAMP '2025-01-02 09:15:00', 'paid', 'west',
   CAST(120.50 AS DECIMAL(12, 2)), ARRAY('priority', 'gift'),
   MAP('device', 'mobile', 'campaign', 'winter')),
  (1002, 2, TIMESTAMP '2025-01-03 11:30:00', 'paid', 'east',
   CAST(80.00 AS DECIMAL(12, 2)), ARRAY('gift'),
   MAP('device', 'web', 'campaign', 'winter')),
  (1003, 1, TIMESTAMP '2025-01-05 16:45:00', 'refund', 'west',
   CAST(-20.00 AS DECIMAL(12, 2)), ARRAY('priority'),
   MAP('device', 'mobile', 'campaign', 'service')),
  (1004, 4, TIMESTAMP '2025-02-01 08:00:00', 'paid', 'north',
   CAST(200.00 AS DECIMAL(12, 2)), ARRAY('new'),
   MAP('device', 'web', 'campaign', 'launch')),
  (1005, CAST(NULL AS BIGINT), TIMESTAMP '2025-02-02 13:00:00',
   'pending', 'south', CAST(NULL AS DECIMAL(12, 2)), ARRAY(),
   MAP('device', 'store'))
AS o(order_id, person_id, event_ts, status, region, amount, tags, attrs)""")
orders = spark.table("guide_orders")
assert orders.count() == 5


## 1.4 View data, schema, and metadata

`show` is bounded display; `printSchema` shows the contract. `columns`,
`dtypes`, `first`, and `isEmpty` support inspection. Remember that actions
can trigger distributed execution.


In [ ]:
orders.printSchema()
print("columns:", orders.columns)
print("dtypes:", orders.dtypes)
print("first:", orders.orderBy("order_id").first().asDict())
print("empty:", orders.isEmpty())


## 1.5 Build lazy transformation pipelines

`select`, `withColumn`, `filter`, and `drop` return new DataFrames. They do
not mutate the original relation. Name derived columns at creation time.


In [ ]:
curated = orders.select(
    "order_id", "person_id", "region", "status", "amount",
).withColumn(
    "reporting_amount", F.coalesce("amount", F.lit(0))
).withColumn(
    "is_positive", F.col("amount") > 0
).filter(
    F.col("status") != "pending"
)
assert curated.count() == 4
curated.orderBy("order_id").show()


## 1.6 DataFrames and temporary tables interoperate

A temporary view exposes a DataFrame to SQL without materializing it.
`spark.table` returns the relation to the fluent API. Views are scoped to the
session unless explicitly created as global temporary views.


In [ ]:
typed_people.createOrReplaceTempView("guide_people")
assert spark.table("guide_people").count() == 3
spark.table("guide_people").groupBy("team").count().orderBy("team").show()


## 1.7 Persistence and plotting are deployment capabilities

`cache`/`persist` trade memory for repeated computation. Native DataFrame
plotting requires Plotly and converts a bounded result to the client. The
Fabric notebook runtime intentionally omits Plotly; a production image can
install it without changing the DataFrame plan. Never plot an unbounded
relation.


In [ ]:
plot_capability = {
    "cache_api": hasattr(orders, "cache"),
    "persist_api": hasattr(orders, "persist"),
    "plot_api": hasattr(orders, "plot"),
    "plotly_available": importlib.util.find_spec("plotly") is not None,
    "bounded_plot_pattern": "orders.limit(1000).plot.bar(...)"
}
assert plot_capability["cache_api"] and plot_capability["plot_api"]
print(plot_capability)


# Chapter 2 — A tour of PySpark data types

Data types are part of query semantics. Width, precision, scale, nullability,
time-zone behavior, and nested element types affect analysis and results.


## 2.1 Primitive types

Boolean, integral, floating, string, binary, date, and timestamp values map
to SQL types. Python integers do not carry a Spark width; the schema does.


In [ ]:
primitives = spark.sql("""SELECT TRUE AS active,
       CAST(7 AS TINYINT) AS tiny,
       CAST(700 AS INT) AS integer_value,
       CAST(7000000000 AS BIGINT) AS long_value,
       CAST(3.5 AS DOUBLE) AS ratio,
       'spark' AS text,
       DATE '2025-01-01' AS event_date,
       TIMESTAMP '2025-01-01 10:30:00' AS event_ts""")
primitives.printSchema()
primitives.show(truncate=False)


## 2.2 Decimal precision versus binary floating point

Use decimal for money and exact contractual quantities. Specify precision
and scale; arithmetic can widen them. Floating point is appropriate for
scientific values where approximate binary representation is expected.


In [ ]:
spark.sql("""SELECT CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) AS floating_sum,
       CAST(0.10 AS DECIMAL(12, 2)) +
         CAST(0.20 AS DECIMAL(12, 2)) AS decimal_sum,
       typeof(CAST(0.10 AS DECIMAL(12, 2))) AS decimal_type""").show(truncate=False)


## 2.3 Dates, timestamps, and intervals

Dates represent calendar days. Timestamps represent instants or local
date-times depending on type and configuration. Calendar intervals preserve
month/day semantics that fixed-second arithmetic cannot.


In [ ]:
orders.select(
    "order_id", "event_ts",
    F.expr("CAST(event_ts AS DATE)").alias("event_date"),
    F.expr("date_trunc('month', event_ts)").alias("month_start"),
    F.expr("event_ts + INTERVAL 2 HOURS").alias("deadline"),
).orderBy("order_id").show()


## 2.4 Arrays, maps, and structs

Complex types preserve structure inside a row. Use typed functions rather
than parsing strings. This endpoint executes collection functions through
expression strings so the server—not the client—resolves function names.


In [ ]:
orders.select(
    "order_id",
    F.expr("size(tags)").alias("tag_count"),
    F.expr("try_element_at(tags, 1)").alias("first_tag"),
    F.expr("element_at(attrs, 'device')").alias("device"),
    F.expr("named_struct('region', region, 'amount', amount)")
     .alias("summary"),
).orderBy("order_id").show(truncate=False)


## 2.5 Cast deliberately

Casts can fail, truncate, round, or return null depending on ANSI settings
and the operation. Validate incoming text before casting and choose
`try_cast` when malformed values should become null rather than fail.


In [ ]:
spark.sql("""SELECT CAST('42' AS INT) AS strict_value,
       TRY_CAST('42' AS INT) AS tolerant_value,
       TRY_CAST('not-a-number' AS INT) AS malformed_value,
       CAST(12.345 AS DECIMAL(8, 2)) AS rounded_decimal""").show()


## 2.6 Semi-structured JSON with an explicit schema

Parse JSON into a typed struct. Schema-on-read catches drift and enables
nested operations. Serialize only at boundaries that require text.


In [ ]:
json_rows = spark.createDataFrame([
    ('{"id":7,"labels":["new","priority"]}',),
], ["raw"])
json_schema = T.StructType([
    T.StructField("id", T.LongType()),
    T.StructField("labels", T.ArrayType(T.StringType())),
])
print("schema contract:", json_schema.simpleString())
json_rows.select(
    F.expr("from_json(raw, 'STRUCT<id: BIGINT, labels: ARRAY<STRING>>')")
     .alias("value")
).show(truncate=False)


# Chapter 3 — Function junction: manipulate structured data

Prefer built-in expressions. They remain visible to the optimizer, avoid
Python serialization, and share semantics across languages.


## 3.1 Clean missing and malformed data

Decide whether to drop, fill, flag, or preserve missing values. Do not fill
every null globally: null may mean unknown, inapplicable, or not yet known.


In [ ]:
cleaned = orders.select(
    "order_id", "status", "amount",
    F.col("amount").isNull().alias("amount_missing"),
    F.coalesce("amount", F.lit(0)).alias("amount_for_reporting"),
)
cleaned.orderBy("order_id").show()


## 3.2 Normalize text and derive categories

Normalize only according to a documented contract. Case folding, trimming,
and regular expressions can alter identifiers and culturally sensitive text.


In [ ]:
orders.select(
    "order_id",
    F.expr("upper(trim(region))").alias("normalized_region"),
    F.when(F.col("amount").isNull(), "unknown")
     .when(F.col("amount") < 0, "reversal")
     .when(F.col("amount") >= 100, "high_value")
     .otherwise("standard").alias("amount_class"),
).orderBy("order_id").show()


## 3.3 Transform arrays with higher-order functions

Higher-order functions transform values inside arrays without exploding row
cardinality. Use full SQL expressions when a client wrapper is unavailable.


In [ ]:
spark.range(1).select(
    F.expr("transform(ARRAY(1,2,3), x -> x * 10)").alias("scaled"),
    F.expr("filter(ARRAY(1,2,3,4), x -> x % 2 = 0)").alias("evens"),
    F.expr("aggregate(ARRAY(1,2,3), 0, (acc, x) -> acc + x)")
     .alias("total"),
).show()


## 3.4 Summarize with several metrics in one pass

Group once and compute all required metrics together. Name outputs and know
whether each aggregate ignores nulls. `COUNT(*)` and `COUNT(column)` differ.


In [ ]:
order_metrics = orders.groupBy("region", "status").agg(
    F.count(F.lit(1)).alias("rows"),
    F.count("amount").alias("known_amounts"),
    F.sum("amount").alias("net_amount"),
    F.round(F.avg("amount"), 2).alias("average_amount"),
    F.min("amount").alias("minimum_amount"),
    F.max("amount").alias("maximum_amount"),
)
order_metrics.orderBy("region", "status").show()


## 3.5 Conditional aggregation for stable reports

Conditional aggregates produce explicit, stable metric columns and often
evolve more safely than dynamic pivots.


In [ ]:
orders.groupBy("region").agg(
    F.sum(F.when(F.col("status") == "paid", F.col("amount"))
          .otherwise(F.lit(0))).alias("paid_amount"),
    F.sum(F.when(F.col("status") == "refund", F.col("amount"))
          .otherwise(F.lit(0))).alias("refund_amount"),
).orderBy("region").show()


## 3.6 Join with explicit multiplicity expectations

Named-key joins merge the key once. Validate uniqueness on the dimension
side before joining; duplicate dimension keys multiply fact rows.


In [ ]:
person_counts = typed_people.groupBy("person_id").count()
assert person_counts.filter(F.col("count") > 1).limit(1).count() == 0
joined = orders.join(typed_people, "person_id", "left")
joined.select(
    "order_id", "person_id", "name", "team", "amount"
).orderBy("order_id").show()


## 3.7 Semi and anti joins express existence

Semi joins retain matching left rows without right columns. Anti joins retain
non-matching left rows. They avoid collecting keys to the driver.


In [ ]:
people_with_orders = typed_people.join(orders, "person_id", "left_semi")
people_without_orders = typed_people.join(orders, "person_id", "left_anti")
people_with_orders.orderBy("person_id").show()
people_without_orders.orderBy("person_id").show()


## 3.8 Deduplicate with an explicit survivor policy

`dropDuplicates` does not specify which full row survives. Rank by recency
and deterministic tie-breakers when the survivor matters.


In [ ]:
updates = spark.createDataFrame([
    (1001, "2025-01-02 09:20:00", "paid", 1),
    (1001, "2025-01-02 09:25:00", "packed", 2),
    (1002, "2025-01-03 11:35:00", "paid", 1),
    (1002, "2025-01-03 11:35:00", "review", 2),
], ["order_id", "updated_at", "state", "priority"])
survivor = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc(), F.col("priority").desc()
)
latest = updates.withColumn(
    "survivor_rank", F.row_number().over(survivor)
).filter("survivor_rank = 1").drop("survivor_rank", "priority")
latest.orderBy("order_id").show()


## 3.9 Window analytics preserve row detail

Windows compute running aggregates, navigation values, and ranks without
collapsing rows. Specify ordering and frame boundaries explicitly.


In [ ]:
running = Window.partitionBy("person_id").orderBy(
    "event_ts", "order_id"
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
ordered = Window.partitionBy("person_id").orderBy("event_ts", "order_id")
orders.select(
    "person_id", "order_id", "amount",
    F.sum("amount").over(running).alias("running_amount"),
    F.lag("amount").over(ordered).alias("previous_amount"),
    F.row_number().over(ordered).alias("sequence"),
).orderBy(F.col("person_id").asc_nulls_last(), "order_id").show()


# Chapter 4 — Bug busting: debugging and observability

Debugging distributed systems starts by classifying the failure: parse,
analysis, optimization, execution, resource pressure, or infrastructure.


## 4.1 Read plans before tuning

`explain` reveals scans, filters, projections, joins, exchanges, aggregates,
sorts, and windows. A plan establishes mechanism; timing establishes cost.


In [ ]:
diagnostic = orders.filter(F.col("status") == "paid").groupBy(
    "region"
).agg(F.sum("amount").alias("total")).orderBy(F.col("total").desc())
diagnostic.explain(mode="extended")


## 4.2 Handle structured analysis errors

Missing columns fail during analysis before data executes. Catch the narrow
PySpark exception, preserve its error class and message, and fix the plan;
do not retry deterministic analysis failures.


In [ ]:
from pyspark.errors import AnalysisException, PySparkException

try:
    orders.select("definitely_missing_column").collect()
except AnalysisException as error:
    print("expected analysis error:", str(error).splitlines()[0][:240])


## 4.3 Bound observations and driver actions

`collect` transfers all rows to the driver. Aggregate or limit first. Record
row counts and representative values so a debug action does not become a
second production outage.


In [ ]:
debug_rows = orders.select("order_id", "status", "amount").orderBy(
    "order_id"
).limit(3).collect()
assert len(debug_rows) == 3
print([row.asDict() for row in debug_rows])


## 4.4 Spark UI, OS tools, profilers, logs, and IDE debugging

The official guide covers Spark UI, `top`/`ps`, profilers, stack traces,
Python worker logging, and IDE debugging. Connect clients observe a remote
server, so process-level tools belong on server/executor hosts. Use notebook
**22_performance_tuning_troubleshooting** and **23_query_xray** for runnable
diagnostics.


In [ ]:
print({
    "spark_ui_url": spark.sparkContext.uiWebUrl,
    "profile_api": hasattr(spark, "profile"),
    "execution_info_api": hasattr(type(orders), "executionInfo"),
    "fabric_monitoring": "Open the notebook Spark details and Fabric monitoring hub",
})


## 4.5 Test schemas, rows, ordering, and edge cases

Test the contract, not just one value: schema, nullability, multiplicity,
deterministic order, empty input, all-null groups, duplicates, and malformed
data. Keep test actions bounded.


In [ ]:
expected = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "value"])
actual = expected.orderBy("id")
assert actual.schema == expected.schema
assert actual.collect() == expected.collect()
assert actual.filter("id < 0").limit(1).count() == 0
print("schema, row, order, and empty-input checks passed")


# Chapter 5 — Unleashing UDFs and UDTFs

Prefer built-ins. Python UDFs cross a serialization boundary and hide logic
from many optimizer rules. UDTFs generate rows and have an even larger
cardinality impact. Use them only when native expressions cannot represent
the operation.


## 5.1 Replace simple UDFs with native expressions

The optimizer understands built-in expressions. This native classification
is portable across clients and avoids Python worker startup and Arrow/pickle
serialization.


In [ ]:
native = orders.withColumn(
    "amount_class",
    F.when(F.col("amount").isNull(), "unknown")
     .when(F.col("amount") < 0, "reversal")
     .when(F.col("amount") >= 100, "high")
     .otherwise("standard"),
)
native.select("order_id", "amount_class").orderBy("order_id").show()


## 5.2 Understand Python UDF capability and cost

PySpark 4.1 distinguishes scalar Python UDFs (row-wise Python objects),
Pandas UDFs (Arrow-serialized pandas Series/DataFrames), and Arrow UDFs
(`pyarrow.Array` values). Related batch APIs include `mapInPandas`,
`applyInPandas`, and `mapInArrow`. Scalar UDF Arrow optimization can be
selected with `useArrow=True` or
`spark.sql.execution.pythonUDF.arrow.enabled`.

All variants require server-side Python worker support. The client API can
exist while the Fabric Spark session was built without its Python bridge, so test
execution—not `hasattr`—during deployment checks. This bounded probe records
this Fabric notebook's expected unsupported result; compatible servers return
`[2, 3]`. Always declare return types and null behavior.


In [ ]:
@F.udf(returnType=T.IntegerType())
def guide_plus_one(value):
    return None if value is None else value + 1

try:
    udf_values = [
        row.value
        for row in spark.range(1, 3).select(
            guide_plus_one("id").alias("value")
        ).collect()
    ]
    python_udf_capability = {"supported": True, "values": udf_values}
except Exception as error:
    python_udf_capability = {
        "supported": False,
        "reason": str(error).splitlines()[0][:240],
        "fallback": "use built-in Column expressions",
    }
print(python_udf_capability)


## 5.3 Understand UDTF cardinality

UDTFs return tables rather than scalar values. The official API supports
Python UDTFs where the server enables server-side registration. This probe
distinguishes client API presence from executable support. Native generators
such as `explode` remain optimizer-visible alternatives for collections.


In [ ]:
exploded = spark.sql("""SELECT order_id, explode(tags) AS tag
  FROM guide_orders
  ORDER BY order_id, tag""")
assert exploded.count() == 5
exploded.show()

@F.udtf(returnType="num: INT")
class GuideCountDown:
    def eval(self, start: int):
        while start > 0:
            yield (start,)
            start -= 1

try:
    spark.udtf.register("guide_count_down", GuideCountDown)
    udtf_values = [
        row.num for row in spark.sql("SELECT * FROM guide_count_down(3)").collect()
    ]
    python_udtf_capability = {"supported": True, "values": udtf_values}
except Exception as error:
    python_udtf_capability = {
        "supported": False,
        "reason": str(error).splitlines()[0][:240],
        "fallback": "use native generators such as explode",
    }
print(python_udtf_capability)


# Chapter 6 — Old SQL, new tricks

SQL and the DataFrame API compile to the same logical-plan system. Choose
the clearest representation for each layer and cross the boundary freely.


## 6.1 Run SQL through PySpark

`spark.sql` returns a DataFrame. SQL is effective for declarative relational
logic, fixtures, CTEs, grouping sets, and syntax-heavy expressions.


In [ ]:
paid_by_region = spark.sql("""SELECT region, COUNT(*) AS orders, SUM(amount) AS total
  FROM guide_orders
  WHERE status = 'paid'
  GROUP BY region""")
paid_by_region.orderBy(F.col("total").desc()).show()


## 6.2 Continue a SQL result with DataFrame APIs

A SQL result is an ordinary DataFrame. Add API transformations without
materializing or converting the data.


In [ ]:
paid_by_region.withColumn(
    "average_order", F.col("total") / F.col("orders")
).orderBy(F.col("average_order").desc()).show()


## 6.3 Expose a DataFrame to SQL

Register a temporary view when a downstream consumer or complex expression
is clearer in SQL. Views store plans, not copies of rows.


In [ ]:
curated.createOrReplaceTempView("guide_curated_orders")
spark.sql("""SELECT region, SUM(reporting_amount) AS reporting_total
  FROM guide_curated_orders
  GROUP BY region
  ORDER BY reporting_total DESC""").show()


## 6.4 Compare SQL and DataFrame plans, not syntax aesthetics

Equivalent SQL and API expressions should produce compatible logical plans.
Use the style that makes invariants obvious, then verify with `explain` and
correctness tests.


In [ ]:
sql_plan = spark.sql("""
SELECT region, SUM(amount) AS total
FROM guide_orders
GROUP BY region
""")
api_plan = orders.groupBy("region").agg(F.sum("amount").alias("total"))
assert sql_plan.orderBy("region").collect() == api_plan.orderBy("region").collect()
print("SQL and DataFrame API results match")


# Chapter 7 — Load and behold: data loading, storage, and formats

Readers and writers are configured builders. Production code specifies
format, schema, options, partitioning, save mode, and data-quality behavior.


## 7.1 Inspect reader and writer surfaces

Readers cover parquet, JSON, CSV, ORC, text, JDBC, and tables. Writers add
format, mode, partitioning, bucketing, and table operations. This shared
notebook avoids mutating persistent storage.


In [ ]:
print({
    "reader": type(spark.read).__name__,
    "read_formats": [name for name in ("parquet", "json", "csv", "orc", "text", "jdbc")
                     if hasattr(spark.read, name)],
    "write_formats": [name for name in ("parquet", "json", "csv", "orc", "saveAsTable")
                      if hasattr(orders.write, name)],
})


## 7.2 Read inline JSON with a schema contract

File examples need external data. The same schema-on-read principles are
demonstrated safely with JSON strings and `from_json`.


In [ ]:
raw = spark.createDataFrame([
    ('{"id":1,"score":9.5}',),
    ('{"id":2,"score":null}',),
], ["raw"])
raw.select(
    F.expr("from_json(raw, 'STRUCT<id: BIGINT, score: DOUBLE>')")
     .alias("record")
).show(truncate=False)


## 7.3 Reader options are format-specific contracts

CSV headers, separators, quoting, malformed-record modes, JSON multiline,
recursive lookup, timestamp formats, and schema merging change semantics.
Keep options close to the read and test malformed data explicitly.


In [ ]:
configured_reader = spark.read.format("csv").options(
    header="true",
    inferSchema="false",
    mode="FAILFAST",
    timestampFormat="yyyy-MM-dd HH:mm:ss",
).schema("id BIGINT, event_ts TIMESTAMP, value DECIMAL(12,2)")
print("configured reader:", type(configured_reader).__name__)


## 7.4 Save modes and idempotence

Writer modes include append, overwrite, ignore, and error-if-exists. Choose
based on an idempotence contract. Partitioning affects layout and pruning;
too many small partitions create a small-file problem. This Fabric notebook's
file handler writes one file and requires a `.parquet` suffix; full Spark
commonly writes a directory of part files. Paths resolve on the remote
server, not necessarily on the Python client's host.


In [ ]:
fabric_io_path = "Files/sparkrust_runtime2_roundtrip.parquet"
try:
    source_df = orders.select("order_id", "amount")
    source_df.write.mode("overwrite").parquet(fabric_io_path)
    io_rows = spark.read.parquet(fabric_io_path).orderBy("order_id").collect()
    assert len(io_rows) == source_df.count()
    print({"fabric_path": fabric_io_path, "rows": len(io_rows)})
finally:
    from notebookutils import mssparkutils
    mssparkutils.fs.rm(fabric_io_path, True)


## 7.5 Governed tables and lakehouse formats

Delta, Iceberg, Hudi, Hive, OneLake, and Unity add transaction, catalog, and
governance semantics beyond file APIs. Use notebooks **20_delta_liquid_clustering**,
**21_federated_catalogs**, **24_lakehouse_detective**, and **25_medallion_bi**
for runnable end-to-end workflows.


In [ ]:
advanced_topics = [
    "Delta Lake optimization",
    "federated catalogs",
    "Lakehouse debugging",
    "medallion architecture",
]
assert len(advanced_topics) == 4
print({"advanced_topics": advanced_topics})


# ANSI Migration Guide — pandas API on Spark

Spark SQL ANSI mode favors explicit, fail-fast semantics. pandas API on
Spark can differ in casts, overflows, duplicate labels, null behavior, and
ordering. Migration requires tests, not only configuration changes.


## 8.1 Detect pandas API and ANSI configuration

pandas-on-Spark is optional and follows the installed pandas/Arrow matrix.
Module discovery alone does not prove version compatibility. This bounded
import probe reports the installed state; this Fabric notebook teaches migration
with SQL/DataFrames when its pandas adapter is incompatible.


In [ ]:
pandas_api_installed = importlib.util.find_spec("pyspark.pandas") is not None
try:
    import pyspark.pandas as ps

    pandas_api_probe = {
        "installed": pandas_api_installed,
        "importable": True,
        "bounded_sum": int(ps.Series([1, 2, 3]).sum()),
    }
except Exception as error:
    pandas_api_probe = {
        "installed": pandas_api_installed,
        "importable": False,
        "reason": str(error).splitlines()[0][:240],
        "migration_path": "validate ANSI behavior with SQL/DataFrames",
    }
print(pandas_api_probe)


## 8.2 Prefer explicit casts under ANSI semantics

`CAST` fails on malformed values under strict behavior; `TRY_CAST` converts
malformed values to null. Choose intentionally and monitor rejected records.


In [ ]:
spark.sql("""SELECT raw,
       TRY_CAST(raw AS INT) AS tolerant_integer,
       TRY_CAST(raw AS INT) IS NOT NULL AS parseable_integer,
       raw IS NULL AS source_missing
  FROM VALUES ('42'), ('-7'), ('oops'), (CAST(NULL AS STRING)) AS v(raw)
  ORDER BY raw NULLS LAST""").show()


## 8.3 Nulls, NaNs, duplicate labels, and ordering require tests

SQL null, IEEE NaN, and pandas missing values are not interchangeable.
Distributed DataFrames have no stable row order without `orderBy`; pandas
workflows often assume order. Make ordering and label uniqueness explicit.


In [ ]:
edge_cases = spark.sql("""SELECT *
  FROM VALUES
  (1, CAST(NULL AS DOUBLE)),
  (2, CAST('NaN' AS DOUBLE)),
  (3, 1.5)
AS edges(id, value)""")
edge_cases.select(
    "id", "value",
    F.col("value").isNull().alias("is_null"),
    F.isnan("value").alias("is_nan"),
).orderBy("id").show()


## 8.4 Migration checklist

PySpark 4.1's official migration surface includes:

- string-number comparison (`1 == '1'` becomes false rather than coercing);
- strict invalid casts (errors instead of silently producing null);
- `MultiIndex.to_series` rows represented as structs/tuples rather than arrays;
- invalid mixed-type operations, including decimal-float arithmetic and
  boolean operations with `None`;
- `spark.sql.ansi.enabled`, the controlling Spark SQL setting;
- `compute.ansi_mode_support`, the pandas-on-Spark compatibility option;
- `compute.fail_on_ansi_mode`, the fail-fast compatibility option.

Before migrating, inventory implicit casts and overflow assumptions; test
malformed text, null versus NaN, duplicate labels, ordering, decimals,
empty/all-null inputs, and both classic and Fabric Runtime 2.0s where supported.


# Capstone — build, inspect, validate, and release a production-style report

The capstone combines schema contracts, joins, nested data, null handling,
grouped metrics, windows, deterministic ordering, bounded collection,
validation, and explicit session cleanup.


In [ ]:
report_base = orders.join(typed_people, "person_id", "left").select(
    "order_id", "person_id", "name", "team", "region", "amount", "attrs"
).withColumn(
    "person", F.coalesce("name", F.lit("Guest"))
).withColumn(
    "device", F.expr("element_at(attrs, 'device')")
)
report = report_base.groupBy("team", "region", "device").agg(
    F.count(F.lit(1)).alias("orders"),
    F.sum("amount").alias("net_amount"),
).withColumn(
    "regional_rank",
    F.dense_rank().over(
        Window.partitionBy("region").orderBy(F.col("net_amount").desc())
    ),
).orderBy(
    F.col("region").asc_nulls_last(), "regional_rank", "team", "device"
)
report.explain(mode="simple")
rows = report.limit(20).collect()
assert rows and sum(row.orders for row in rows) == 5
for row in rows:
    print(row.asDict())
# Fabric owns the Spark session; do not call spark.stop().


# Production checklist and next steps

- Treat schemas, nullability, precision, and time zones as contracts.
- Prefer built-in expressions over Python UDFs.
- Validate join multiplicity before aggregating.
- Use deterministic deduplication and ordering.
- Aggregate or limit before collecting.
- Inspect plans before tuning; measure under matched resources.
- Test malformed, empty, duplicate, skewed, and all-null inputs.
- Make writes idempotent and catalog-aware.
- Keep classic-driver-only and Connect-compatible code paths explicit.
- Continue with the Spark API, Spark SQL, streaming, ML, lakehouse, and
  troubleshooting notebooks for deeper specialist references.
